# L2 — Pluggable Inference Layer

## What you'll learn

- How to build a provider-agnostic inference layer using LiteLLM
- How to configure a model per agent role: sub-agents first, then the orchestrator
- How to swap LLM backends by changing a single line
- How to publish model configuration to SSM for downstream layers

## Overview

Every AnyCompany agent needs a language model to think and respond. This notebook sets up that model layer so the choice of model is never built into the agents.

The agents reach their models through [LiteLLM](https://docs.litellm.ai/), an open source library that talks to many model providers in the same way. Switching an agent from Amazon Bedrock to Baseten, or from one model to another, is a configuration change. The agent code stays the same.

There are two agent roles, and each gets its own model:

- **Specialist agents** (Order, Refund and Research) handle one kind of request each.
- **The orchestrator** talks to the customer, decides which specialist to call, and puts the answer together.

You pick a provider and model for each role in the **Partner Selection UI** (Model Provider). The UI gives you the code to paste below. You set up and test the specialists' model first, then the orchestrator's, and finally save both choices so the agents in L3 can use them.

## Architecture

![Model inference layer: the specialist agents and the orchestrator both call their models through LiteLLM, which routes to Amazon Bedrock by default or to Baseten as the partner option.](images/l2-model-inference-architecture.png)

The L3 notebooks read the saved choices when they deploy the agents. To change a model later, re-run this notebook and then L3. The agent code doesn't change.

## Steps

0. Install the packages
1. Set up LiteLLM
2. **Specialist agents:** choose the model (code from the Partner Selection UI)
3. **Specialist agents:** test the model
4. **Orchestrator:** choose the model (code from the Partner Selection UI)
5. **Orchestrator:** test the model
6. Save the model choices

## License details

Refer to LICENSE file in the main folder for license details.

## Prerequisites

- The Partner Selection UI from L0 ([`1_partner_selection.ipynb`](../l0-partner-selection/1_partner_selection.ipynb)) is running.
- You don't need to finish L1 ([`1_knowledge_base.ipynb`](../l1-data-knowledge/1_knowledge_base.ipynb)) first: this notebook doesn't depend on the data layer.

## Step 0: Install the packages

Install LiteLLM and the AWS SDK. Run this once, at the start.


In [ ]:
!pip install -q litellm==1.83.7 --upgrade boto3


## Step 1: Set up LiteLLM

This setup is the same for every provider. It sets the AWS Region and defines the helper that sends a prompt to a model through LiteLLM. Run it before the code you paste from the Partner Selection UI.


In [ ]:
# Shared LiteLLM setup — constant across all roles (not provider-swappable).

import os
import time
import litellm

# Route Bedrock calls to the right region for every LiteLLM call.
REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
os.environ["AWS_REGION_NAME"] = REGION
print("REGION:", REGION)


class InferenceBackend:
    """Pluggable inference backend powered by LiteLLM."""
    def __init__(self, model: str, system_prompt: str = None,
                 max_tokens: int = 1024, temperature: float = None):
        self.model = model
        self.system_prompt = system_prompt
        self.max_tokens = max_tokens
        # None = the model's own default. Newer models (Claude Sonnet 5, Opus 5,
        # Haiku 5.5, GPT-5.6) reject the temperature parameter, so it is only
        # sent when you set it explicitly.
        self.temperature = temperature
    @property
    def name(self) -> str:
        return self.model
    def invoke(self, prompt: str, **kwargs) -> dict:
        """Invoke the model and return the response text and latency."""
        messages = []
        if self.system_prompt:
            messages.append({"role": "system", "content": self.system_prompt})
        messages.append({"role": "user", "content": prompt})
        start = time.time()
        params = {"max_tokens": kwargs.get("max_tokens", self.max_tokens)}
        temperature = kwargs.get("temperature", self.temperature)
        if temperature is not None:
            params["temperature"] = temperature
        resp = litellm.completion(model=self.model, messages=messages, **params)
        latency = time.time() - start
        return {"content": resp.choices[0].message.content, "latency": latency}


## Step 2: Specialist agents — choose the model

In the **Partner Selection UI** (Model Provider), pick a model for the **Specialist Agents** and copy the generated code. Paste it into the empty cell below and run it. This sets the model that the Order, Refund and Research agents will use.


<div style="border-left: 6px solid #6d5cff; background-color: #f1efff; padding: 10px 14px; border-radius: 6px; margin: 4px 0;">
<b>🧩 Partner Selection UI: paste a snippet into the cell below</b>
<ol style="margin: 6px 0 0 0;">
<li>In the Partner Selection UI, open <b>Inference → Model Provider</b>.</li>
<li>Select the <b>Specialist Agents</b> tab, then pick a provider and a model. For a partner provider, save its API key.</li><li>Expand <b>📋 Specialist Agents code snippet</b> and copy the code.</li>
<li>Paste it into the empty cell below, then run the cell.</li>
</ol>
</div>

In [ ]:
# 👉 Paste the "📋 Specialist Agents code snippet" from the Partner Selection UI (Inference → Model Provider) here, then run.



## Step 3: Specialist agents — test the model

Send a sample question to the specialists' model and check that it answers.

In [ ]:
# Test the sub-agent backend: send a sample prompt and print response + latency.
for q in ["What is AWS?"]:
    print(f"\n{'='*60}")
    print(f"Customer: {q}")
    print(f"Sub-agent model ({active_backend.name}):")
    result = active_backend.invoke(q)
    print(result["content"])
    print(f"\n  ⏱ Latency: {result['latency']:.2f}s")

## Step 4: Orchestrator — choose the model

In the **Partner Selection UI** (Model Provider), pick a model for the **Orchestrator** and copy the generated code. Paste it into the empty cell below and run it.

A Bedrock model needs no key. A Baseten model uses the API key you saved in the UI. The model for each role is saved for L3 in the final step.

<div style="border-left: 6px solid #6d5cff; background-color: #f1efff; padding: 10px 14px; border-radius: 6px; margin: 4px 0;">
<b>🧩 Partner Selection UI: paste a snippet into the cell below</b>
<ol style="margin: 6px 0 0 0;">
<li>In the Partner Selection UI, open <b>Inference → Model Provider</b>.</li>
<li>Select the <b>Orchestrator Agent</b> tab, then pick a provider and a model. For a partner provider, save its API key.</li><li>Expand <b>📋 Orchestrator Agent code snippet</b> and copy the code.</li>
<li>Paste it into the empty cell below, then run the cell.</li>
</ol>
</div>

In [ ]:
# 👉 Paste the "📋 Orchestrator Agent code snippet" from the Partner Selection UI (Inference → Model Provider) here, then run.



## Step 5: Orchestrator — test the model

Send a sample question to the orchestrator's model and check that it answers.

In [ ]:
# Test the orchestrator model via LiteLLM. harness_model_id is a LiteLLM id
# (bedrock/... or baseten/...), the same shape L3 passes to the AgentCore Harness
# as liteLlmModelConfig. harness_api_base is only set for a provider with a custom
# endpoint; the Step 4 snippet set the API key for this test call.
# Self-contained: defines its own prompt so this cell runs independently.
import time
harness_api_base = globals().get("harness_api_base")
messages = [
    {"role": "system", "content": "You are a customer support orchestrator that "
                                   "plans and delegates to specialist agents."},
    {"role": "user", "content": "A customer asks for a refund and an order status. "
                                 "In one sentence, how would you delegate this?"},
]
start = time.time()
resp = litellm.completion(
    model=harness_model_id,
    messages=messages,
    max_tokens=256,
    **({"api_base": harness_api_base} if harness_api_base else {}),
)
print(f"Orchestrator model ({harness_model_id}):")
print(resp.choices[0].message.content)
print(f"\n  ⏱ Latency: {time.time()-start:.2f}s")

## Step 6: Save the model choices

Save the model for each role so the L3 notebooks can use them when they deploy the agents. Run this last, after both models have passed their tests.


In [ ]:
# Publish all model IDs to SSM Parameter Store (run after configuring all roles).
# model_id / harness_model_id were set in Steps 2 and 4.
import boto3
ssm = boto3.client("ssm", region_name=os.environ["AWS_REGION_NAME"])
harness_api_base = globals().get("harness_api_base")
params = {
    "/anycompany/agentcore/model_id": active_backend.model,   # sub-agents
    "/anycompany/agentcore/harness_model_id": harness_model_id,   # orchestrator (LiteLLM id)
}
for key, value in params.items():
    ssm.put_parameter(Name=key, Value=value, Type="String", Overwrite=True)
    print(f"Set {key} = {value}")
# Orchestrator endpoint: published for a partner model, removed for Bedrock so a
# stale endpoint from an earlier run never reaches the Harness.
HARNESS_API_BASE_PARAM = "/anycompany/agentcore/harness_api_base"
if harness_api_base:
    ssm.put_parameter(Name=HARNESS_API_BASE_PARAM, Value=harness_api_base, Type="String", Overwrite=True)
    print(f"Set {HARNESS_API_BASE_PARAM} = {harness_api_base}")
else:
    try:
        ssm.delete_parameter(Name=HARNESS_API_BASE_PARAM)
        print(f"Removed {HARNESS_API_BASE_PARAM} (Bedrock needs no endpoint)")
    except ssm.exceptions.ParameterNotFound:
        pass

## Summary

The specialist agents and the orchestrator each have a saved model, chosen from Amazon Bedrock or Baseten. L3 reads these choices when it deploys the agents.

To change a model later, choose a different one in the Partner Selection UI, paste and run its code in Step 2 or Step 4, run Step 6 again, and re-run L3.

**Next:** [`1_gateway_tools.ipynb`](../l3-orchestration/1_gateway_tools.ipynb). Add the tools the agents will call.

---
## Cleanup
When you are done with the workshop, uncomment and run the cell below to delete the model choices saved by this notebook and any partner API keys saved in the Partner Selection UI.


In [ ]:
## === CLEANUP: Delete the SSM parameters created by this notebook ===
## ⚠ Uncomment and run this cell to clean up this run's SSM parameters.
## Deletes both the model id params (plain Strings) AND any partner API-key
## params (SecureStrings) saved by the L0 Partner Selection UI.
# import boto3, os
#
# REGION = os.environ.get("AWS_REGION_NAME", os.environ.get("AWS_DEFAULT_REGION", "us-east-1"))
# ssm = boto3.client("ssm", region_name=REGION)
#
# PARAMS = [
#     # Model ids written by Step 6
#     "/anycompany/agentcore/model_id",
#     "/anycompany/agentcore/harness_model_id",
#     "/anycompany/agentcore/harness_api_base",
#     # Partner API keys saved by the L0 UI (safe if not present)
#     "/anycompany/agentcore/config/inference/subagents/api_key",
#     "/anycompany/agentcore/config/inference/orchestrator/api_key",
# ]
#
# for name in PARAMS:
#     try:
#         ssm.delete_parameter(Name=name)
#         print(f"Deleted SSM parameter: {name}")
#     except ssm.exceptions.ParameterNotFound:
#         pass  # already deleted or never set (e.g. Bedrock-only run)
#     except Exception as e:
#         print(f"  Failed to delete {name}: {e}")
#
# print("\n✅ All L2 SSM parameters cleaned up.")